# A1 · Bagging from scratch: bootstrap, vote and out-of-bag error

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Implement bagging of trees with numpy and scikit-learn (only `DecisionTreeClassifier` as a building block): bootstrap samples, majority vote and **out-of-bag error** computed by hand, checking that it matches the estimate of `BaggingClassifier`. Also measure that a bootstrap sample contains 63.2% of the distinct examples.

## Dataset

`make_classification(2000, 12 features, 5 informative, 2 redundant, flip_y=0.08)`, 60/40 split: `X_tr, y_tr, X_te, y_te`.

## Educational purpose

Take the mystery out of the random forest: it is a loop of bootstraps, trees and votes. Understand how the OOB error gives a validation estimate **without a separate set**, and why the vote of 100 trees beats each single tree by a wide margin.

## Task

1. `bagging(X, y, B, seed)`: for `b` in `range(B)` draw bootstrap indices (`rng.integers(0, n, n)`), train `DecisionTreeClassifier(random_state=b)` and return the list `trees` and the list `indices` (the bootstrap indices).
2. `vote_predict(trees, X)`: predictions by majority vote (mean of the tree predictions > 0.5).
3. `oob(trees, indices, X, y)`: for each training example collect the votes **only of the trees whose bootstrap does not contain it**; return `(oob_predictions, covered)` where `covered` is the mask of the examples with at least one OOB tree. Compute `oob_acc` on the covered set.
4. Save `single_acc` (mean test accuracy of the 100 trees), `vote_acc` (test accuracy of the vote), `distinct_share` (mean fraction of distinct examples in the bootstraps) and `covered_share` (fraction of examples with at least one OOB tree: with 100 trees it is almost 1). Compare `oob_acc` with `BaggingClassifier(DecisionTreeClassifier(), 100, oob_score=True, random_state=0).oob_score_`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(2000, n_features=12, n_informative=5, n_redundant=2, class_sep=0.8, flip_y=0.08, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)

## Your work

In [ ]:
def bagging(X, y, B, seed=0):
    ...


def vote_predict(trees, X):
    ...


def oob(trees, indices, X, y):
    ...

In [ ]:
trees, indices = bagging(X_tr, y_tr, 100)
single_acc = None
vote_acc = None
distinct_share = None

In [ ]:
oob_predictions, covered = oob(trees, indices, X_tr, y_tr)
oob_acc = None
covered_share = None
sklearn_acc = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=0).fit(X_tr, y_tr).oob_score_

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='OOB votes: `votes = np.zeros(n); n_votes = np.zeros(n)`; for each tree `outside = np.setdiff1d(np.arange(n), indices[b])`; `votes[outside] += tree.predict(X[outside]); n_votes[outside] += 1`; prediction `votes / n_votes > 0.5` where `n_votes > 0`.'):
    check(len(trees) == 100 and len(indices) == 100 and all(len(i) == len(y_tr) for i in indices), "100 trees and 100 bootstrap samples of the same size as the training set", "rng.integers(0, n, n)")
    check(0.62 < distinct_share < 0.645, "a bootstrap contains on average 63.2% of distinct examples", "len(np.unique(i)) / n")
    check(vote_acc > single_acc + 0.06 and single_acc < 0.8 and vote_acc > 0.82, "the vote (over 0.82) beats the single tree (below 0.8) by over 6 points", "mean of the predictions > 0.5")
    check(covered_share > 0.99 and oob_predictions.shape == y_tr.shape, "with 100 trees almost all the examples have at least one OOB tree", "an example is covered if it is missing from the bootstrap of at least one tree")
    check(abs(oob_acc - vote_acc) < 0.05 and abs(oob_acc - sklearn_acc) < 0.02, "the OOB accuracy is close to the test one (within 5 points) and to the estimate of BaggingClassifier (within 2)", "only trees that have not seen the example")
    check(oob_acc < 0.999, "the OOB is not 1.0: it is not the estimate on the training set (which is almost perfect)", "an example never votes with the trees that have seen it")
done()

---
The commented solution is in `solutions/A_from_scratch/A1_bagging_and_oob_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.